# VERITAS OpenForensics — Compare / epoch-1 Test

This notebook does **not** train.

It uses existing `predictions/<exp>_predictions.csv` when present. If a variant only
has a checkpoint, it **scores official Test here** from `{exp}_last.pt` (else `_best.pt`).

**Notebooks 2 and 3 are optional.** Baseline + Full is enough. One variant is OK if
you only want that epoch-1 Test number.

Attach:

1. `defactodataset/defactoface` (needed only when a CSV is missing)
2. `ahmedgamal/openforensics-wild-dataset` (same extra reals as training, when scoring from a checkpoint)
3. Each training notebook’s latest output (epoch-1 is fine)

Use **GPU T4** if any checkpoint must be scored. CSV-only compare can run on CPU.

- `baseline_effb7_defacto`
- `multistream_no_seg_defacto` (optional)
- `seg_no_multistream_defacto` (optional)
- `full_veritas_defacto`


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from scipy import stats
except Exception:
    stats = None

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())

SEARCH_ROOTS = []
if Path("/kaggle/input").exists():
    SEARCH_ROOTS.append(Path("/kaggle/input"))
if Path("/kaggle/working").exists():
    SEARCH_ROOTS.append(Path("/kaggle/working"))
SEARCH_ROOTS.append(Path.cwd())
SEARCH_ROOTS.append(Path.cwd() / "kaggle_working")

EXPECTED = [
    "baseline_effb7_defacto",
    "multistream_no_seg_defacto",
    "seg_no_multistream_defacto",
    "full_veritas_defacto",
]
print("search roots:", [str(p) for p in SEARCH_ROOTS if p.exists()])


In [ ]:
def find_files(name_substr: str, suffix: str):
    hits = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() == suffix and name_substr in p.name:
                hits.append(p)
    return sorted(set(hits))


pred_files = {}
ckpt_files = {}
for exp in EXPECTED:
    cands = find_files(f"{exp}_predictions", ".csv")
    pred_files[exp] = cands[0] if cands else None
    lasts = [p for p in find_files(f"{exp}_last", ".pt") if p.name == f"{exp}_last.pt"]
    bests = [p for p in find_files(f"{exp}_best", ".pt") if p.name == f"{exp}_best.pt"]
    ckpt_files[exp] = (lasts[0] if lasts else None) or (bests[0] if bests else None)
    print(exp, "predictions:", pred_files[exp])
    print(exp, "checkpoint:", ckpt_files[exp])

protocol_files = find_files("protocol", ".json")
print("protocol files", [str(p) for p in protocol_files])
NEED_INFER = [e for e in EXPECTED if pred_files[e] is None and ckpt_files[e] is not None]
print("will score Test from checkpoint:", NEED_INFER)


In [ ]:
protocols = []
for p in protocol_files:
    try:
        protocols.append((p, json.loads(p.read_text(encoding="utf-8"))))
    except Exception as e:
        print("skip protocol", p, e)

if protocols:
    hashes = {obj.get("hash") for _, obj in protocols}
    print("PROTOCOL hashes found:", hashes)
    if len(hashes) > 1:
        print("WARNING: protocol hashes differ. Do not treat metrics as a controlled ablation.")
        for p, obj in protocols:
            print(" ", p, obj.get("hash"), obj.get("experiment"))
    else:
        print("OK: all protocol.json files share the same hash.")
else:
    print("No protocol.json found. Comparison can still run if prediction file names match.")


## Score official Test from epoch-1 checkpoints (only if a CSV is missing)


In [ ]:
import hashlib
import json as _json
import os
import cv2
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B7_Weights, efficientnet_b7
from tqdm.auto import tqdm

META = {
    "baseline_effb7_defacto": {"multi_stream": False, "segmentation": False, "legacy": True},
    "multistream_no_seg_defacto": {"multi_stream": True, "segmentation": False, "legacy": False},
    "seg_no_multistream_defacto": {"multi_stream": False, "segmentation": True, "legacy": False},
    "full_veritas_defacto": {"multi_stream": True, "segmentation": True, "legacy": False},
}
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
(OUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "logs").mkdir(parents=True, exist_ok=True)


def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


EXTS_IMG = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def discover_defacto_root() -> Path:
    roots = []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.extend([Path.cwd() / "defacto-face", Path.cwd() / "dataset", Path.cwd()])
    for root in roots:
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            kids = {d.lower() for d in dirnames}
            if "swapping_img" in kids or "morphing_img" in kids:
                return p
    raise FileNotFoundError("Attach defactodataset/defactoface to score Test from checkpoints.")


def _dir(root, name):
    if root is None or not Path(root).is_dir():
        return None
    root = Path(root)
    if (root / name).is_dir():
        return root / name
    for p in root.iterdir():
        if p.is_dir() and p.name.lower() == name.lower():
            return p
    return None


def _as_img_dir(p):
    if p is None or not Path(p).is_dir():
        return None
    p = Path(p)
    return p / "img" if (p / "img").is_dir() else p


def _norm_stem(stem: str) -> str:
    s = stem.lower()
    for tok in ("_probe_mask", "_probemask", "_probe", "_mask", "_gt", "-mask"):
        s = s.replace(tok, "")
    return s


def _id_key(stem: str) -> str:
    digits = "".join(ch for ch in _norm_stem(stem) if ch.isdigit())
    return digits.lstrip("0") or digits


def index_mask_dir(mask_root):
    idx = {}
    if mask_root is None or not Path(mask_root).is_dir():
        return idx
    mask_root = Path(mask_root)
    skip_meta = {".json", ".txt", ".xml", ".csv", ".md"}
    files = [p for p in mask_root.rglob("*") if p.is_file()]
    image_files = [p for p in files if p.suffix.lower() in EXTS_IMG] or [p for p in files if p.suffix.lower() not in skip_meta]
    for p in image_files:
        idx[p.stem] = p
        idx[p.stem.lower()] = p
        idx[_norm_stem(p.stem)] = p
        kid = _id_key(p.stem)
        if kid:
            idx[kid] = p
        parent = p.parent
        if parent != mask_root and parent.name.lower() not in {"img", "probe_mask", "masks", "images"}:
            idx[parent.name] = p
            idx[parent.name.lower()] = p
    return idx


def lookup_mask(image_path: Path, img_dir, mask_idx):
    keys = [image_path.stem, image_path.stem.lower(), _norm_stem(image_path.stem), _id_key(image_path.stem)]
    try:
        rel = image_path.relative_to(img_dir).with_suffix("").as_posix().lower()
        keys.extend([rel, Path(rel).name])
    except ValueError:
        pass
    for k in keys:
        if k and k in mask_idx:
            return mask_idx[k]
    return None


def assign_split(key: str, seed=42) -> str:
    h = hashlib.md5(f"{seed}:{key}".encode()).hexdigest()
    x = int(h[:8], 16) / 0xFFFFFFFF
    if x < 0.70:
        return "train"
    if x < 0.85:
        return "val"
    return "test"


def list_test(root: Path, seed=42) -> pd.DataFrame:
    rows = []
    swap_img = _as_img_dir(_dir(root, "swapping_img"))
    morph_img = _as_img_dir(_dir(root, "morphing_img"))
    ref_img = _as_img_dir(_dir(root, "reference"))
    swap_mask = _as_img_dir(_dir(_dir(root, "swapping_annotations"), "probe_mask"))
    morph_mask = _as_img_dir(_dir(_dir(root, "morphing_annotations"), "probe_mask"))
    sources = []
    if swap_img is not None:
        sources.append((swap_img, swap_mask))
    if morph_img is not None:
        sources.append((morph_img, morph_mask))
    for img_dir, mask_dir in sources:
        imgs = sorted([p for p in img_dir.rglob("*") if p.is_file() and p.suffix.lower() in EXTS_IMG])
        mask_idx = index_mask_dir(mask_dir)
        paired = []
        unmatched = []
        for p in imgs:
            if assign_split(p.stem, seed) != "test":
                continue
            mp = lookup_mask(p, img_dir, mask_idx)
            if mp is None:
                unmatched.append(p)
            else:
                paired.append((p, mp))
        if not paired and unmatched and mask_dir is not None:
            masks = sorted([p for p in Path(mask_dir).rglob("*") if p.is_file() and p.suffix.lower() in EXTS_IMG])
            if len(masks) == len(imgs):
                paired = [(p, m) for p, m in zip(imgs, masks) if assign_split(p.stem, seed) == "test"]
                unmatched = []
        for p, mp in paired:
            rows.append({"path": str(p), "mask_path": str(mp), "label": 1, "frame_idx": 0, "mask_frame_idx": 0, "file_name": p.name})
    if ref_img is not None:
        for p in sorted(ref_img.rglob("*")):
            if not (p.is_file() and p.suffix.lower() in EXTS_IMG):
                continue
            if assign_split("real:" + p.stem, seed) != "test":
                continue
            rows.append({"path": str(p), "mask_path": "", "label": 0, "frame_idx": 0, "mask_frame_idx": 0, "file_name": p.name})
    inp = Path("/kaggle/input")
    if inp.is_dir():
        for child in sorted(inp.iterdir()):
            if not child.is_dir() or "defacto" in child.name.lower():
                continue
            for dirpath, dirnames, _ in os.walk(child):
                p = Path(dirpath)
                if p.parent.name.lower() != "test":
                    continue
                lab = 0 if p.name.lower() == "real" else 1 if p.name.lower() == "fake" else None
                if lab is None:
                    continue
                for img in sorted([q for q in p.rglob("*") if q.is_file() and q.suffix.lower() in EXTS_IMG]):
                    rows.append({"path": str(img), "mask_path": "", "label": lab, "frame_idx": 0, "mask_frame_idx": 0, "file_name": img.name})
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    rng = np.random.RandomState(seed + 23)
    real = df[df.label == 0]
    gt_fake = df[(df.label == 1) & (df.mask_path.fillna("") != "")]
    extra_fake = df[(df.label == 1) & (df.mask_path.fillna("") == "")]
    if len(real) > 2000:
        real = real.sample(n=2000, random_state=rng)
    if len(gt_fake) > 2000:
        gt_fake = gt_fake.sample(n=2000, random_state=rng)
    if len(extra_fake) > 2000:
        extra_fake = extra_fake.sample(n=2000, random_state=rng)
    return pd.concat([real, gt_fake, extra_fake]).reset_index(drop=True)

IMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD_NP = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def letterbox_rgb(image_rgb, size):
    h, w = image_rgb.shape[:2]
    scale = min(size / max(h, 1), size / max(w, 1))
    nw, nh = max(int(round(w * scale)), 1), max(int(round(h * scale)), 1)
    resized = cv2.resize(image_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    left, top = (size - nw) // 2, (size - nh) // 2
    canvas[top : top + nh, left : left + nw] = resized
    return canvas, (left, top, nw, nh)


def letterbox_mask(mask, size, box):
    left, top, nw, nh = box
    canvas = np.zeros((size, size), dtype=np.float32)
    if mask is None or mask.size == 0:
        return canvas
    m = mask.astype(np.float32)
    m = (m > 127).astype(np.float32) if m.max() > 1.0 else (m > 0.5).astype(np.float32)
    canvas[top : top + nh, left : left + nw] = cv2.resize(m, (nw, nh), interpolation=cv2.INTER_NEAREST)
    return canvas


try:
    _HAAR = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
except Exception:
    _HAAR = cv2.CascadeClassifier()


def read_frame(path: Path, idx: int):
    if path.is_file() and path.suffix.lower() in EXTS_IMG:
        bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        return None if bgr is None else cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if path.is_dir():
        frames = sorted([c for c in path.iterdir() if c.is_file() and c.suffix.lower() in EXTS_IMG])
        if not frames:
            return None
        idx = int(np.clip(idx, 0, len(frames) - 1))
        bgr = cv2.imread(str(frames[idx]), cv2.IMREAD_COLOR)
        return None if bgr is None else cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(max(idx, 0)))
    ok, bgr = cap.read()
    cap.release()
    return None if (not ok or bgr is None) else cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def bbox_from_mask(mask, margin):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    h, w = mask.shape[:2]
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    pad_x = int(round(max(x1 - x0, 1) * margin))
    pad_y = int(round(max(y1 - y0, 1) * margin))
    return max(0, x0 - pad_x), max(0, y0 - pad_y), min(w, x1 + pad_x), min(h, y1 + pad_y)


class DEFACTOFaceDataset(Dataset):
    def __init__(self, df, size, margin=0.10):
        self.df = df.reset_index(drop=True)
        self.size = int(size)
        self.margin = float(margin)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rgb = read_frame(Path(row.path), int(row.frame_idx))
        if rgb is None:
            rgb = np.zeros((self.size, self.size, 3), dtype=np.uint8)
        is_fake = int(row.label) == 1
        mask_np = None
        if is_fake and str(getattr(row, "mask_path", "") or ""):
            raw = read_frame(Path(row.mask_path), int(row.mask_frame_idx if "mask_frame_idx" in row.index else row.frame_idx))
            if raw is not None:
                mask_np = cv2.cvtColor(raw, cv2.COLOR_RGB2GRAY) if raw.ndim == 3 else raw
        if mask_np is not None:
            mask_bin = (mask_np > 127).astype(np.uint8) if mask_np.max() > 1 else (mask_np > 0).astype(np.uint8)
            box = bbox_from_mask(mask_bin, self.margin)
        else:
            mask_bin = np.zeros(rgb.shape[:2], dtype=np.uint8)
            gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
            faces = _HAAR.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(48, 48)) if not _HAAR.empty() else []
            if len(faces):
                x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
                dummy = np.zeros(rgb.shape[:2], dtype=np.uint8)
                dummy[y : y + h, x : x + w] = 1
                box = bbox_from_mask(dummy, self.margin)
            else:
                box = None
        if box is None:
            box = (0, 0, rgb.shape[1], rgb.shape[0])
        if not is_fake:
            mask_bin = np.zeros(rgb.shape[:2], dtype=np.uint8)
        x0, y0, x1, y1 = box
        crop, mcrop = rgb[y0:y1, x0:x1], mask_bin[y0:y1, x0:x1]
        canvas, lbox = letterbox_rgb(crop, self.size)
        mask = letterbox_mask(mcrop, self.size, lbox) if mask_np is not None else np.zeros((self.size, self.size), dtype=np.float32)
        x = np.transpose((canvas.astype(np.float32) / 255.0 - IMAGENET_MEAN_NP) / IMAGENET_STD_NP, (2, 0, 1))
        return {
            "image": torch.from_numpy(x.copy()),
            "mask": torch.from_numpy(mask[None].copy()),
            "label": torch.tensor([float(row.label)], dtype=torch.float32),
            "has_gt_mask": torch.tensor([1.0 if mask_np is not None else 0.0], dtype=torch.float32),
            "path": row.path,
            "file_name": row.file_name,
        }


def _srm_kernels():
    k1 = np.array([[0, 0, 0, 0, 0], [0, -1, 2, -1, 0], [0, 2, -4, 2, 0], [0, -1, 2, -1, 0], [0, 0, 0, 0, 0]], dtype=np.float32)
    k2 = np.array([[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2], [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]], dtype=np.float32)
    k3 = np.array([[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 1, -2, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], dtype=np.float32)
    ks = []
    for k in (k1, k2, k3):
        s = np.abs(k).sum()
        ks.append(k / (s if s > 0 else 1.0))
    return torch.from_numpy(np.stack(ks, 0)[:, None, :, :])


class MultiStreamExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("srm_weight", _srm_kernels())

    def forward(self, x_rgb_norm):
        mean, std = IMAGENET_MEAN.to(x_rgb_norm.device, x_rgb_norm.dtype), IMAGENET_STD.to(x_rgb_norm.device, x_rgb_norm.dtype)
        rgb = (x_rgb_norm * std + mean).clamp(0, 1)
        y = 0.299 * rgb[:, 0:1] + 0.587 * rgb[:, 1:2] + 0.114 * rgb[:, 2:3]
        residual = F.conv2d(y, self.srm_weight.to(dtype=rgb.dtype), padding=2).tanh()
        spec = torch.fft.fftshift(torch.fft.fft2(rgb, norm="ortho"), dim=(-2, -1))
        mag = torch.log1p(spec.abs())
        b, c, h, w = mag.shape
        mag_flat = mag.reshape(b, c, -1)
        mn, mx = mag_flat.min(dim=-1, keepdim=True).values, mag_flat.max(dim=-1, keepdim=True).values
        mag = ((mag_flat - mn) / (mx - mn + 1e-6)).reshape(b, c, h, w)
        return torch.cat([rgb, residual, mag], dim=1)


class FeatureFusion(nn.Module):
    def __init__(self, conv3):
        super().__init__()
        conv9 = nn.Conv2d(9, conv3.out_channels, kernel_size=conv3.kernel_size, stride=conv3.stride, padding=conv3.padding, bias=False)
        with torch.no_grad():
            w = conv3.weight
            conv9.weight[:, 0:3].copy_(w)
            conv9.weight[:, 3:6].copy_(w)
            conv9.weight[:, 6:9].copy_(w)
            conv9.weight.mul_(1.0 / 3.0)
        self.conv = conv9

    def forward(self, x):
        return self.conv(x)


class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch=256, rates=(6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList()
        self.branches.append(nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)))
        for r in rates:
            self.branches.append(nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)))
        gn = 32 if out_ch % 32 == 0 else 1
        self.gap = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.GroupNorm(gn, out_ch), nn.ReLU(inplace=True))
        self.project = nn.Sequential(nn.Conv2d(out_ch * 5, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.Dropout(0.1))

    def forward(self, x):
        res = [b(x) for b in self.branches]
        gap = F.interpolate(self.gap(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        res.append(gap)
        return self.project(torch.cat(res, dim=1))


class DeepLabV3PlusHead(nn.Module):
    def __init__(self, high_ch, low_ch, num_classes=1):
        super().__init__()
        self.aspp = ASPP(high_ch, 256)
        self.low_proj = nn.Sequential(nn.Conv2d(low_ch, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        self.decoder = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, 1),
        )

    def forward(self, high, low, out_size):
        high = F.interpolate(self.aspp(high), size=low.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([high, self.low_proj(low)], dim=1)
        return F.interpolate(self.decoder(x), size=out_size, mode="bilinear", align_corners=False)


class ClassificationHead(nn.Module):
    def __init__(self, in_ch, dropout=0.5):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_ch, 1))

    def forward(self, feat):
        return self.fc(self.pool(feat).flatten(1))


class BaselineEffNet(nn.Module):
    def __init__(self):
        super().__init__()
        net = efficientnet_b7(weights=None)
        self.features = net.features
        self.cls_head = ClassificationHead(2560)
        self.multi_stream = False
        self.segmentation = False

    def forward(self, x):
        logits = self.cls_head(self.features(x))
        return {"classification_logits": logits, "fake_probability": torch.sigmoid(logits)}


class EfficientNetBackbone(nn.Module):
    def __init__(self, multi_stream=False):
        super().__init__()
        net = efficientnet_b7(weights=None)
        stem = net.features[0]
        self.stem_rest = nn.Sequential(*list(stem.children())[1:])
        conv0 = stem[0]
        self.multi_stream = multi_stream
        self.stream = MultiStreamExtractor() if multi_stream else None
        if multi_stream:
            self.fusion = FeatureFusion(conv0)
            self.stem_conv = self.fusion.conv
        else:
            self.fusion = None
            self.stem_conv = conv0
        self.stages = nn.ModuleList(list(net.features.children())[1:])
        self.out_channels = 2560
        self.low_stage, self.high_stage = 1, 4

    def forward(self, x):
        x = self.stem_conv(self.stream(x) if self.multi_stream else x)
        x = self.stem_rest(x)
        low = high = None
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if i == self.low_stage:
                low = x
            if i == self.high_stage:
                high = x
        return {"final": x, "low": low if low is not None else x, "high": high if high is not None else x}


class VERITASModel(nn.Module):
    def __init__(self, multi_stream=False, segmentation=False):
        super().__init__()
        self.multi_stream = bool(multi_stream)
        self.segmentation = bool(segmentation)
        self.encoder = EfficientNetBackbone(multi_stream=self.multi_stream)
        self.cls_head = ClassificationHead(self.encoder.out_channels)
        self.seg_head = DeepLabV3PlusHead(224, 48, 1) if self.segmentation else None

    def forward(self, x):
        feats = self.encoder(x)
        logits = self.cls_head(feats["final"])
        out = {"classification_logits": logits, "fake_probability": torch.sigmoid(logits)}
        if self.segmentation:
            seg_logits = self.seg_head(feats["high"], feats["low"], out_size=x.shape[-2:])
            out["segmentation_logits"] = seg_logits
            out["segmentation_prob"] = torch.sigmoid(seg_logits)
        return out


def build_eval_model(exp):
    cfg = META[exp]
    if cfg["legacy"]:
        return BaselineEffNet()
    return VERITASModel(multi_stream=cfg["multi_stream"], segmentation=cfg["segmentation"])


@torch.no_grad()
def score_loader(model, loader, device, thresh=0.5):
    model.eval()
    recs = []
    for batch in tqdm(loader, desc="test"):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].numpy().reshape(-1).astype(int)
        out = model(images)
        prob = out["fake_probability"].detach().cpu().numpy().reshape(-1)
        pred = (prob >= thresh).astype(int)
        for i in range(len(labels)):
            recs.append({
                "file_name": batch["file_name"][i],
                "path": batch["path"][i],
                "split": "test",
                "ground_truth_label": int(labels[i]),
                "predicted_label": int(pred[i]),
                "fake_probability": float(prob[i]),
            })
    return recs


if NEED_INFER:
    if not torch.cuda.is_available():
        print("WARNING: scoring checkpoints on CPU will be slow. Settings → GPU T4.")
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DEFACTO_ROOT = discover_defacto_root()
    ck0 = _torch_load(ckpt_files[NEED_INFER[0]])
    cfg0 = ck0.get("config") or {}
    image_size = int(cfg0.get("image_size") or 600)
    seed = int(cfg0.get("seed") or 42)
    print("DEFACTO Test root", DEFACTO_ROOT, "image_size", image_size)
    test_df = list_test(DEFACTO_ROOT, seed=seed)
    print("Test images:", len(test_df))
    batch_size = 4
    loader = DataLoader(DEFACTOFaceDataset(test_df, image_size), batch_size=batch_size, shuffle=False, num_workers=2 if os.name != "nt" else 0)
    for exp in NEED_INFER:
        print("SCORING", exp, ckpt_files[exp])
        ck = _torch_load(ckpt_files[exp])
        model = build_eval_model(exp).to(DEVICE)
        model.load_state_dict(ck["model_state_dict"])
        recs = score_loader(model, loader, DEVICE)
        dest = OUT_DIR / "predictions" / f"{exp}_predictions.csv"
        pd.DataFrame(recs).to_csv(dest, index=False)
        pred_files[exp] = dest
        acc = float((pd.DataFrame(recs).ground_truth_label == pd.DataFrame(recs).predicted_label).mean()) if recs else float("nan")
        print(exp, "epoch", ck.get("epoch"), "TEST acc", acc, "wrote", dest)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("All variants already have prediction CSVs. Skipping checkpoint Test.")


In [ ]:
frames = {}
for exp, path in pred_files.items():
    if path is None:
        print("MISSING predictions for", exp)
        continue
    df = pd.read_csv(path)
    need = {"file_name", "ground_truth_label", "predicted_label", "fake_probability"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns {missing}")
    df["experiment"] = exp
    frames[exp] = df
    print(exp, "n=", len(df), "acc=", float((df.ground_truth_label == df.predicted_label).mean()))

if len(frames) < 1:
    raise RuntimeError("Need at least one prediction CSV or a checkpoint to score.")
if len(frames) < 2:
    print("Only one variant scored. Table will print; McNemar needs two.")


In [ ]:
def cls_metrics(df):
    y = df["ground_truth_label"].astype(int)
    p = df["predicted_label"].astype(int)
    cm = confusion_matrix(y, p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    row = {
        "n_faces": int(len(df)),
        "accuracy": float(accuracy_score(y, p)),
        "f1": float(f1_score(y, p, zero_division=0)),
        "precision": float(precision_score(y, p, zero_division=0)),
        "recall": float(recall_score(y, p, zero_division=0)),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }
    if "iou" in df.columns:
        fake = df[(df.ground_truth_label == 1) & df["iou"].notna()]
        row["miou_fake_only"] = float(fake["iou"].mean()) if len(fake) else float("nan")
        row["dice_fake_only"] = float(fake["dice"].mean()) if len(fake) and "dice" in fake.columns else float("nan")
    else:
        row["miou_fake_only"] = "N/A"
        row["dice_fake_only"] = "N/A"
    return row


META = {
    "baseline_effb7_defacto": {"multi_stream": False, "segmentation": False},
    "multistream_no_seg_defacto": {"multi_stream": True, "segmentation": False},
    "seg_no_multistream_defacto": {"multi_stream": False, "segmentation": True},
    "full_veritas_defacto": {"multi_stream": True, "segmentation": True},
}

rows = []
for exp in EXPECTED:
    if exp not in frames:
        rows.append({"experiment": exp, "status": "NOT RUN", **META[exp]})
        continue
    row = {"experiment": exp, "status": "ok", **META[exp], **cls_metrics(frames[exp])}
    rows.append(row)

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / "defacto_compare_summary.csv", index=False)
print("wrote", out_dir / "defacto_compare_summary.csv")


In [ ]:
def mcnemar(a: pd.DataFrame, b: pd.DataFrame):
    ka = {str(r.file_name): r for r in a.itertuples()}
    kb = {str(r.file_name): r for r in b.itertuples()}
    common = sorted(set(ka) & set(kb))
    n01 = n10 = 0
    for k in common:
        a_ok = int(ka[k].ground_truth_label) == int(ka[k].predicted_label)
        b_ok = int(kb[k].ground_truth_label) == int(kb[k].predicted_label)
        if a_ok and not b_ok:
            n01 += 1
        elif (not a_ok) and b_ok:
            n10 += 1
    n = n01 + n10
    if stats is None:
        p = None
    else:
        p = 1.0 if n == 0 else float(stats.binomtest(min(n01, n10), n=n, p=0.5).pvalue)
    return {"n_paired": len(common), "a_correct_b_wrong": n01, "a_wrong_b_correct": n10, "pvalue": p}


pairs = [
    ("baseline_effb7_defacto", "multistream_no_seg_defacto"),
    ("baseline_effb7_defacto", "seg_no_multistream_defacto"),
    ("baseline_effb7_defacto", "full_veritas_defacto"),
    ("multistream_no_seg_defacto", "full_veritas_defacto"),
    ("seg_no_multistream_defacto", "full_veritas_defacto"),
]
stat_rows = []
for a, b in pairs:
    if a not in frames or b not in frames:
        stat_rows.append({"pair": f"{a} vs {b}", "status": "missing"})
        continue
    mc = mcnemar(frames[a], frames[b])
    stat_rows.append({"pair": f"{a} vs {b}", "mcnemar": mc})
    print(f"{a} vs {b}")
    print("  McNemar", mc)

(out_dir / "defacto_statistical_tests.json").write_text(json.dumps(stat_rows, indent=2, default=str), encoding="utf-8")
print("wrote", out_dir / "defacto_statistical_tests.json")


## How to read this

- Epoch-1 is a valid compare: attach those outputs even if Test was skipped in training.
- If `PROTOCOL_HASH` differed, stop and rerun the mismatched notebook with the agreed CONFIG (`full`, seed 42, size 600).
- Pairing is by `file_name` on the 15% DEFACTO test split (same hash protocol).
- `miou_fake_only` / `dice_fake_only` exist only for the two segmentation variants and score **probe_mask GT**.
- This table is not comparable to Wild (`notebooks-wild/WILD_05_compare.ipynb`).
